# CNN và attention: xem từng phép tính, rồi kiểm chứng

Đi cùng ngày **22** và **25** trên web. Dự đoán trước, tiến một bước, ghi lại điều khác với dự đoán. Tất cả dữ liệu/vector/trọng số ở đây là minh họa đặt tay. Các bài này tính **forward**, chưa huấn luyện CNN/Transformer.

## 1. CNN nhìn ảnh theo cửa sổ

Ảnh được biểu diễn bằng ma trận pixel. Một bộ lọc dùng chung ở mọi vị trí. Thư viện CNN thường tính cross-correlation (không lật kernel); tên convolution vẫn thường dùng. ReLU là max(0,z). Max pooling 2×2 stride 2 lấy cực đại, bỏ phần cuối không đủ khối.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def cnn(image,kernel,stride=1,padding=0,bias=0):
    image=np.asarray(image,float);kernel=np.asarray(kernel,float)
    padded=np.pad(image,padding);h=(padded.shape[0]-kernel.shape[0])//stride+1;w=(padded.shape[1]-kernel.shape[1])//stride+1
    out=np.zeros((h,w));windows=[]
    for i in range(h):
        for j in range(w):
            patch=padded[i*stride:i*stride+kernel.shape[0],j*stride:j*stride+kernel.shape[1]]
            products=patch*kernel;out[i,j]=products.sum()+bias
            windows.append({'row':i,'col':j,'patch':patch,'products':products,'sum':out[i,j]})
    return out,windows

def pool2(image):
    image=np.asarray(image);h,w=image.shape
    return np.array([[image[2*i:2*i+2,2*j:2*j+2].max() for j in range(w//2)] for i in range(h//2)])
image=np.zeros((5,5));image[:,2]=1
kernel=np.array([[-1,0,1],[-1,0,1],[-1,0,1.]])
stride=1;padding=0;bias=0
output,windows=cnn(image,kernel,stride,padding,bias)
print('Kích thước feature map:',output.shape)

In [ ]:
window=0  # đổi vị trí cửa sổ: 0 tới len(windows)-1
selected=windows[window]
display(pd.DataFrame({'pixel':selected['patch'].ravel(),'weight':kernel.ravel(),'product':selected['products'].ravel()}))
print('Tổng tích + bias:',selected['products'].sum(),'+',bias,'=',selected['sum'])
relu=np.maximum(output,0);pooled=pool2(relu)
fig,axs=plt.subplots(1,5,figsize=(14,3))
for ax,values,title in zip(axs,[image,kernel,output,relu,pooled],['Input','Fixed kernel','Before ReLU','After ReLU','Max pool 2x2']):
    im=ax.imshow(values,cmap='RdBu_r');ax.set_title(title);fig.colorbar(im,ax=ax,shrink=.6)
plt.tight_layout();plt.show()

### Thực hành CNN A/B/C

A. Tính tay chín tích tại cửa sổ đầu và kiểm tra dấu trước/sau ReLU.

B. Chỉ đổi pixel hàng 1 cột 1. Dự đoán những ô output nào đổi, rồi chạy bên dưới. Có cửa sổ chứa pixel nhưng trọng số tương ứng bằng 0 nên output không đổi không?

C. So stride 1/2 và padding 0/1. Giải thích công thức floor((H+2P−K)/S)+1. Pooling có khôi phục được mọi pixel không? Bộ lọc đặt sẵn khác bộ lọc được học như thế nào?

In [ ]:
modified=image.copy();modified[0,0]=1-modified[0,0]
new_output,_=cnn(modified,kernel,stride,padding,bias)
print('Chênh lệch feature map khi đổi một pixel:');display(pd.DataFrame(new_output-output))
for s in [1,2]:
    for p in [0,1]:
        values,_=cnn(image,kernel,s,p,bias)
        print('stride',s,'padding',p,'shape',values.shape)
assert np.array_equal(pool2(np.array([[1,2,3],[9,4,7],[8,6,5]])),[[9]])

## 2. Attention: mỗi token tổng hợp thông tin từ các token khác

Các hàng theo thứ tự **Em, học, AI, hôm nay**. Đây là vector giả lập, chưa phải embedding học từ tiếng Việt. Q=XWq, K=XWk, V=XWv. Score=QKᵀ/√dₖ; softmax theo từng hàng; output=weights V. Causal mask chặn key ở tương lai. Temperature là tham số thí nghiệm thêm; attention chuẩn dùng T=1.

Attention weight chỉ là hệ số trong phép tính này, không tự chứng minh quan hệ ngôn ngữ hoặc giải thích nhân quả.

In [ ]:
tokens=['Em','học','AI','hôm nay']
X=np.array([[1,0],[0,1],[1,1],[-1,.5]])
Wq=np.array([[1,.5],[0,1]]);Wk=np.array([[.5,1],[1,0]]);Wv=np.array([[1,-1],[.5,1]])
def attend(X,Wq,Wk,Wv,causal=False,temperature=1):
    Q=X@Wq;K=X@Wk;V=X@Wv
    dots=Q@K.T;scores=dots/np.sqrt(Q.shape[1])/temperature
    if causal:scores=np.where(np.triu(np.ones_like(scores,dtype=bool),1),-np.inf,scores)
    exponent=np.exp(scores-np.max(scores,axis=1,keepdims=True));weights=exponent/exponent.sum(axis=1,keepdims=True)
    return dict(Q=Q,K=K,V=V,dots=dots,scores=scores,weights=weights,output=weights@V,scale=np.sqrt(Q.shape[1]))
causal=False;temperature=1;query=0
result=attend(X,Wq,Wk,Wv,causal,temperature)
for key in ['Q','K','V']:print(key);display(pd.DataFrame(result[key],index=tokens))

In [ ]:
display(pd.DataFrame({'token':tokens,'dot':result['dots'][query],'scaled_masked':result['scores'][query],'weight':result['weights'][query]}))
contributions=result['weights'][query,:,None]*result['V']
display(pd.DataFrame(contributions,index=tokens,columns=['output dim 1 contribution','output dim 2 contribution']))
print('Cộng từng chiều:',contributions.sum(axis=0),'= output:',result['output'][query])
fig,axs=plt.subplots(1,2,figsize=(9,4))
for ax,field in zip(axs,['weights','output']):
    im=ax.imshow(result[field],cmap='Blues',vmin=0 if field=='weights' else None);ax.set_yticks(range(4),tokens);ax.set_title(field);fig.colorbar(im,ax=ax)
axs[0].set_xticks(range(4),tokens);plt.tight_layout();plt.show()
assert np.allclose(result['weights'].sum(axis=1),1)
assert np.allclose(contributions.sum(axis=0),result['output'][query])

### Thực hành attention A/B/C

A. Chọn một query và một key, tính tay từng tích Q·K, rồi chia √2.

B. Bật causal mask: query đầu tiên còn đọc được token nào? Tại sao output hàng đầu bằng V hàng đầu?

C. Chỉ đổi temperature hoặc độ lớn vector AI. So phân bố trọng số và output, giữ nguyên W. Không kết luận attention weight cao nghĩa là token đó là nguyên nhân của dự đoán. Multi-head, positional encoding, residual, LayerNorm và huấn luyện Transformer chưa nằm trong ví dụ này.

In [ ]:
masked=attend(X,Wq,Wk,Wv,True)
assert np.allclose(masked['output'][0],masked['V'][0])
assert np.all(masked['weights'][np.triu_indices(4,1)]==0)
display(pd.DataFrame(masked['weights'],index=tokens,columns=tokens))
for temp in [.25,1,2]:
    value=attend(X,Wq,Wk,Wv,False,temp)
    print('T=',temp,'weights query đầu:',np.round(value['weights'][0],4),'output:',np.round(value['output'][0],4))

## 3. Kiểm chứng JSON tải từ web

Lưu JSON cùng thư mục notebook, sửa TRACE_PATH. CNN kiểm tra mọi cửa sổ/pixel/trọng số/tích; attention kiểm tra toàn bộ Q/K/V, dots, mask, softmax và output. File CNN chứa toàn bộ phép tính dù trên web mới quan sát một phần. Score −∞ trong JSON được ghi bằng chuỗi `-Infinity`.

In [ ]:
TRACE_PATH='' # ví dụ 'trace-cnn.json' hoặc 'trace-attention.json'
def audit_tensor(trace):
    if trace['algorithm']=='cnn':
        image=np.array(trace['image']);kernel=np.array(trace['kernel']);s=trace['stride'];p=trace['padding'];b=trace['bias']
        output,windows=cnn(image,kernel,s,p,b);expected=trace['result']
        assert np.allclose(output,expected['output'])
        assert output.shape==(expected['height'],expected['width'])
        assert len(windows)==len(expected['windows'])
        for win,record in zip(windows,expected['windows']):
            assert win['row']==record['row'] and win['col']==record['col']
            assert np.isclose(win['sum'],record['sum']) and np.isclose(max(0,win['sum']),record['relu'])
            for n,term in enumerate(record['terms']):
                ky,kx=divmod(n,kernel.shape[1]);y=win['row']*s+ky-p;x=win['col']*s+kx-p
                inside=0<=y<image.shape[0] and 0<=x<image.shape[1]
                assert term['y']==y and term['x']==x and term['inside']==inside
                assert np.isclose(term['pixel'],image[y,x] if inside else 0)
                assert np.isclose(term['weight'],kernel[ky,kx]) and np.isclose(term['product'],win['products'][ky,kx])
    else:
        assert trace['algorithm']=='attention'
        actual=attend(np.array(trace['X']),np.array(trace['Wq']),np.array(trace['Wk']),np.array(trace['Wv']),trace['causal'],trace['temperature'])
        for key,value in actual.items():
            expected=np.array(trace['result'][key],dtype=float)
            assert np.allclose(value,expected),key
    print('Khớp NumPy:',trace['algorithm'])
if TRACE_PATH:audit_tensor(json.loads(Path(TRACE_PATH).read_text()))
else:print('Chưa chọn trace; các phần trên vẫn chạy độc lập.')

## Nhật ký học

Em đã dự đoán đúng phép tính nào? Pixel/token nào làm output thay đổi? Cấu hình nào chỉ đổi cách tính forward, cấu hình nào sẽ cần huấn luyện lại trong mô hình thật? Hai ví dụ chưa cung cấp bằng chứng gì về độ chính xác trên dữ liệu mới?